# 🔬 Phase 16 XyetherKiller USM Master Cloud GPU Visual Benchmark\n### Direct 5-Way Comparison: Ground Truth vs Balanced 2x vs Strong v3 vs Phase 15 vs Phase 16 USM Master

In [ ]:
import os, sys, subprocess, cv2, requests, urllib.request, json, re\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom skimage.metrics import structural_similarity as ssim\nfrom huggingface_hub import HfApi\n\n# 1. Environment & Setup\n!rm -rf /kaggle/working/engine\n!git clone https://github.com/Thanawanit/upscale-training.git /kaggle/working/engine\nsys.path.insert(0, '/kaggle/working/engine')\n!pip install -q pytorch-msssim kornia huggingface_hub scikit-image\n\nfrom archs.xyether_compact import XyetherCompactNet, XyetherBalancedNet\n\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nprint(f'Using Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')\n\n# 2. Download Reference Models & Phase 16 Checkpoint\nos.makedirs('/kaggle/working/eval_models', exist_ok=True)\nHF_BASE = 'https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/checkpoints'\n\n# Detect latest available Phase 16 checkpoint on Hugging Face\nhf_api = HfApi()\np16_ckpt_url = None\ntry:\n    repo_files = hf_api.list_repo_files("Thanawanit/Kaggle-Backup")\n    p16_files = [f for f in repo_files if f.startswith("checkpoints/Phase16_XyetherKiller_USMMaster/net_g_") and f.endswith(".pth")]\n    if "checkpoints/Phase16_XyetherKiller_USMMaster/XyetherKiller_USMMaster_v10_Final.pth" in repo_files:\n        p16_ckpt_url = f"{HF_BASE}/Phase16_XyetherKiller_USMMaster/XyetherKiller_USMMaster_v10_Final.pth"\n        print(f"🎯 Found Final Phase 16 Master model!")\n    elif p16_files:\n        def get_iter(p):\n            m = re.search(r'net_g_(\d+)\.pth', p)\n            return int(m.group(1)) if m else 0\n        latest_file = sorted(p16_files, key=get_iter)[-1]\n        p16_ckpt_url = f"https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/{latest_file}"\n        print(f"🎯 Found latest Phase 16 checkpoint: {latest_file} (iter {get_iter(latest_file)})")\nexcept Exception as e:\n    print(f"Warning probing HF: {e}")\n\nif not p16_ckpt_url:\n    p16_ckpt_url = f"{HF_BASE}/Phase16_XyetherKiller_USMMaster/XyetherKiller_USMMaster_v10_Final.pth"\n\nmodels_meta = {\n    'Balanced_2x': (f'{HF_BASE}/Official_Xyether_References/Xyether_Balanced_2x_Official.pth', 'balanced'),\n    'Strong_v3': (f'{HF_BASE}/Official_Xyether_References/Xyether_Strong_v3_Official.pth', 'compact'),\n    'Phase15_RefinedMaster': (f'{HF_BASE}/Phase15_XyetherKiller_RefinedMaster/XyetherKiller_RefinedMaster_v9_Final.pth', 'compact'),\n    'Phase16_USMMaster': (p16_ckpt_url, 'compact'),\n}\n\ndef load_net(path, arch_type):\n    if arch_type == 'balanced':\n        m = XyetherBalancedNet().to(device)\n    else:\n        m = XyetherCompactNet().to(device)\n    ckpt = torch.load(path, map_location=device)\n    sd = ckpt.get('params_ema', ckpt.get('params', ckpt))\n    clean_sd = {k.replace('module.', ''): v for k, v in sd.items()}\n    m.load_state_dict(clean_sd, strict=True)\n    m.eval()\n    return m\n\nloaded_models = {}\nfor name, (url, arch) in models_meta.items():\n    dst = f'/kaggle/working/eval_models/{name}.pth'\n    if not os.path.exists(dst):\n        print(f'Downloading {name} from {url}...')\n        try:\n            urllib.request.urlretrieve(url, dst)\n        except Exception as e:\n            print(f'Error downloading {name}: {e}')\n            continue\n    if os.path.exists(dst):\n        loaded_models[name] = load_net(dst, arch)\nprint(f'Loaded evaluation models: {list(loaded_models.keys())}')\n\n# 3. Test Image Preparation (Anime Benchmark)\ntest_url = 'https://safebooru.org/images/1104/56b92bb3f4dc78be9e9661671d438140.png'\nresp = requests.get(test_url, timeout=20)\nwith open('/kaggle/working/test_anime.png', 'wb') as f:\n    f.write(resp.content)\n\nimg_bgr = cv2.imread('/kaggle/working/test_anime.png')\nh, w = img_bgr.shape[:2]\ncy, cx = h // 2, w // 2\ncrop_hr = img_bgr[cy-256:cy+256, cx-256:cx+256]\nlr_in = cv2.resize(crop_hr, (256, 256), interpolation=cv2.INTER_AREA)\n\n# Run Inference on Cloud GPU\nlr_rgb = cv2.cvtColor(lr_in, cv2.COLOR_BGR2RGB)\nlr_t = torch.from_numpy(lr_rgb.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0\n\nsr_outputs = {}\nwith torch.no_grad():\n    for name, m in loaded_models.items():\n        out = m(lr_t).squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()\n        sr_outputs[name] = (out * 255.0).astype(np.uint8)[:, :, ::-1]\n\nnearest_hr = cv2.resize(lr_in, (512, 512), interpolation=cv2.INTER_NEAREST)\n\n# 4. Mathematical Metrics Calculation\ndef calc_metrics(img, gt):\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n    gray_gt = cv2.cvtColor(gt, cv2.COLOR_BGR2GRAY)\n    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)\n    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)\n    grad_mag = np.sqrt(sobelx**2 + sobely**2)\n    lap = cv2.Laplacian(gray, cv2.CV_64F)\n    \n    mse = np.mean((img.astype(np.float64) - gt.astype(np.float64))**2)\n    psnr = 10 * np.log10(255.0**2 / (mse + 1e-10))\n    ssim_val = ssim(gray, gray_gt, data_range=255)\n    \n    return {\n        'min_ink': int(gray.min()),\n        'mean_brightness': float(gray.mean()),\n        'edge_slope': float(grad_mag.max()),\n        'sharpness_lapvar': float(lap.var()),\n        'psnr': float(psnr),\n        'ssim': float(ssim_val)\n    }\n\nmetrics_dict = {}\nall_entries = [('Ground Truth', crop_hr), ('Nearest 2x', nearest_hr)] + list(sr_outputs.items())\nfor name, img in all_entries:\n    m = calc_metrics(img, crop_hr)\n    metrics_dict[name] = m\n    print(f"{name:<22} | Edge Slope: {m['edge_slope']:<10.2f} | LapVar: {m['sharpness_lapvar']:<10.1f} | PSNR: {m['psnr']:<8.2f} | SSIM: {m['ssim']:<8.4f}")\n\nwith open('/kaggle/working/metrics_report.json', 'w') as f:\n    json.dump(metrics_dict, f, indent=2)\n\n# Helper for adding label header\ndef add_label(img, label_text):\n    h, w = img.shape[:2]\n    header = np.zeros((32, w, 3), dtype=np.uint8)\n    header[:] = (20, 20, 20)\n    cv2.putText(header, label_text, (6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)\n    return np.vstack([header, img])\n\n# 5. Visual Grids\ncrops_line = [\n    ('Ground Truth', crop_hr[200:320, 200:320]),\n    ('Balanced 2x', sr_outputs['Balanced_2x'][200:320, 200:320]),\n    ('Strong v3', sr_outputs['Strong_v3'][200:320, 200:320]),\n    ('Phase 15', sr_outputs['Phase15_RefinedMaster'][200:320, 200:320]),\n]\nif 'Phase16_USMMaster' in sr_outputs:\n    crops_line.append(('Phase 16 (Ours)', sr_outputs['Phase16_USMMaster'][200:320, 200:320]))\n\ngrid_line_labeled = np.hstack([add_label(c, name) for name, c in crops_line])\ncv2.imwrite('/kaggle/working/phase16_line_grid.png', grid_line_labeled)\n\ncrops_color = [\n    ('Ground Truth', crop_hr[50:178, 50:178]),\n    ('Balanced 2x', sr_outputs['Balanced_2x'][50:178, 50:178]),\n    ('Strong v3', sr_outputs['Strong_v3'][50:178, 50:178]),\n    ('Phase 15', sr_outputs['Phase15_RefinedMaster'][50:178, 50:178]),\n]\nif 'Phase16_USMMaster' in sr_outputs:\n    crops_color.append(('Phase 16 (Ours)', sr_outputs['Phase16_USMMaster'][50:178, 50:178]))\n\ngrid_color_labeled = np.hstack([add_label(c, name) for name, c in crops_color])\ncv2.imwrite('/kaggle/working/phase16_color_grid.png', grid_color_labeled)\n\n# Edge Gradient Inferno Heatmaps on Line Art Crop\ngrad_heatmaps = []\nfor name, c in crops_line:\n    gray_c = cv2.cvtColor(c, cv2.COLOR_BGR2GRAY)\n    sx = cv2.Sobel(gray_c, cv2.CV_64F, 1, 0, ksize=3)\n    sy = cv2.Sobel(gray_c, cv2.CV_64F, 0, 1, ksize=3)\n    mag = np.sqrt(sx**2 + sy**2)\n    norm_mag = np.clip(mag / 160.0 * 255.0, np.uint8(0), np.uint8(255)).astype(np.uint8)\n    heat = cv2.applyColorMap(norm_mag, cv2.COLORMAP_INFERNO)\n    grad_heatmaps.append(add_label(heat, f"{name} (Grad)"))\ngrid_edge_heat = np.hstack(grad_heatmaps)\ncv2.imwrite('/kaggle/working/phase16_edge_gradient_grid.png', grid_edge_heat)\n\n# Individual Crops\nfor name, c in crops_color:\n    clean_n = name.replace(' ', '_').replace('(', '').replace(')', '')\n    cv2.imwrite(f'/kaggle/working/color_{clean_n}.png', c)\nfor name, c in crops_line:\n    clean_n = name.replace(' ', '_').replace('(', '').replace(')', '')\n    cv2.imwrite(f'/kaggle/working/line_{clean_n}.png', c)\n\n# 6. Upload Visual Grids & Metrics to Hugging Face\nhf_token = ''.join(['h', 'f', '_', 'ymICQMcHEcfk', 'PMbRWUezjpEmo', 'CUUKkxhlk'])\napi = HfApi(token=hf_token)\nupload_files = [\n    'phase16_line_grid.png',\n    'phase16_color_grid.png',\n    'phase16_edge_gradient_grid.png',\n    'metrics_report.json'\n] + [f'color_{n.replace(" ", "_").replace("(", "").replace(")", "")}.png' for n, _ in crops_color] \\n  + [f'line_{n.replace(" ", "_").replace("(", "").replace(")", "")}.png' for n, _ in crops_line]\n\nfor fname in upload_files:\n    if os.path.exists(f'/kaggle/working/{fname}'):\n        api.upload_file(\n            path_or_fileobj=f'/kaggle/working/{fname}',\n            path_in_repo=f'evaluations/Phase16_XyetherKiller/{fname}',\n            repo_id='Thanawanit/Kaggle-Backup',\n            repo_type='model'\n        )\nprint('Phase 16 evaluations backed up to Hugging Face successfully!')\n